#Contents
The quantization tables are computed in the same way as the original `0_extract_quant_tables` file.

The difference is in the last steps: the `TRANSITION_STEPS` are set to quality = 1, with a multiplicative factor that leads to the last step, in which only the DC is preserved. In this step, all the values corresponding thìo the AC are set to big numbers. The transition steps allow to gradually go from a very low quality to the last step.

In [ ]:
! pip install jpegio

In [ ]:
from PIL import Image
import numpy as np
import jpegio
import matplotlib.pyplot as plt
import os
import torch

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
BASE_DIR = "/content/drive/MyDrive/Colab Notebooks/tesi/" if 'google.colab' in str(get_ipython()) else "."
sample_jpeg_path = os.path.join(BASE_DIR, "experiments/tables/reference_image/000001.jpg")

In [ ]:
SAVE_PATH = os.path.join(BASE_DIR, "assets/dequant-quant2_qt.pt")

## computing quantization tables

In [ ]:
# qualities = [100, 25, 15, 10, 8, 7, 6, 5, 4, 3, 1]
# qualities = list(range(100, -1, -10))

qualities = [100, 15, 11, 8, 5, 3, 2, 1, 1, 1]

In [ ]:
current_img = Image.open(sample_jpeg_path)
quant_tables = {}

for i, q in enumerate(qualities):
    current_img.save(f'output_img_quality_{i}.jpg', quality=q)
    current_img = Image.open(f'output_img_quality_{i}.jpg').convert('RGB')
    jpeg_data = jpegio.read(f'output_img_quality_{i}.jpg')
    quant_tables[f'step{i}_luminance'] = np.array(jpeg_data.quant_tables[0])
    quant_tables[f'step{i}_chrominance'] = np.array(jpeg_data.quant_tables[1])

Here are reported various techniques to implement the transition. Among these, the one that was visually better is the linear boost.

In [ ]:
TRANSITION_STEPS = 3
n_steps = len(qualities)

for idx in range(n_steps):
    for suffix in ['luminance', 'chrominance']:
        key = f'step{idx}_{suffix}'

        if 'quant_tables' in globals() and isinstance(quant_tables, dict) and key in quant_tables:
            # Retrieve the numpy array and convert it to a torch.Tensor for operations
            value = torch.tensor(quant_tables[key], dtype=torch.float32).detach().clone()

            steps_from_end = n_steps - 1 - idx

            if steps_from_end < TRANSITION_STEPS:
                # Apply the boosting logic for the last TRANSITION_STEPS
                # The boost factor increases as we get closer to the end.

                # Option 0: exponential (powers of 2)
                # boost = 2 ** (TRANSITION_STEPS - steps_from_end)

                # Option 1: Linear boost (lighter)
                boost = (TRANSITION_STEPS - steps_from_end) + 0.5

                # Option 2: Smaller base for exponential boost (e.g., base 1.5)
                # boost = 1.5 ** (TRANSITION_STEPS - steps_from_end)

                # Option 3: Additive boost (adds a constant or linearly increasing value)
                # boost = 1.0 + (TRANSITION_STEPS - steps_from_end) * 0.5 # Example: adds 0.5, 1.0, 1.5

                value = value * boost

                # Assuming this specific modification is still desired for the boosted tables.
                if value.numel() > 0:
                    value[0, 0] = 1.0

            # Update the quantization table in the dictionary (now as a torch.Tensor)
            quant_tables[key] = value
        else:
            print(f"Key '{key}' not found in quant_tables or quant_tables is not defined/a dict.")

printing quantization tables

In [ ]:
for i in range(num_images):
    saved_img = jpegio.read(f'output_img_quality_{i}.jpg')
    for (quant_table, original_table) in zip(saved_img.quant_tables, jpeg_data.quant_tables):
        print(f"Quality {qualities[i]}:\n{quant_table}")

Quality 100:
[[1 1 1 1 1 1 1 1]
 [1 1 1 1 1 1 1 1]
 [1 1 1 1 1 1 1 1]
 [1 1 1 1 1 1 1 1]
 [1 1 1 1 1 1 1 1]
 [1 1 1 1 1 1 1 1]
 [1 1 1 1 1 1 1 1]
 [1 1 1 1 1 1 1 1]]
Quality 100:
[[1 1 1 1 1 1 1 1]
 [1 1 1 1 1 1 1 1]
 [1 1 1 1 1 1 1 1]
 [1 1 1 1 1 1 1 1]
 [1 1 1 1 1 1 1 1]
 [1 1 1 1 1 1 1 1]
 [1 1 1 1 1 1 1 1]
 [1 1 1 1 1 1 1 1]]
Quality 15:
[[ 53  37  33  53  80 133 170 203]
 [ 40  40  47  63  87 193 200 183]
 [ 47  43  53  80 133 190 230 186]
 [ 47  57  73  97 170 255 255 206]
 [ 60  73 123 186 226 255 255 255]
 [ 80 117 183 213 255 255 255 255]
 [163 213 255 255 255 255 255 255]
 [240 255 255 255 255 255 255 255]]
Quality 15:
[[ 57  60  80 157 255 255 255 255]
 [ 60  70  87 220 255 255 255 255]
 [ 80  87 186 255 255 255 255 255]
 [157 220 255 255 255 255 255 255]
 [255 255 255 255 255 255 255 255]
 [255 255 255 255 255 255 255 255]
 [255 255 255 255 255 255 255 255]
 [255 255 255 255 255 255 255 255]]
Quality 11:
[[ 73  50  45  73 109 182 232 255]
 [ 54  54  64  86 118 255 255 250]


# saving quantization tables

### pt

scelgo pt perchè mi permette di avere i file già pronti per la GPU quando uso torch-dct

In [ ]:
import torch

torch.save(quant_tables, SAVE_PATH)
print(f"Quantization tables saved to: {SAVE_PATH}")

Quantization tables saved to: /content/drive/MyDrive/Colab Notebooks/tesi/assets/dequant-quant2_qt.pt


Verying that data have been saved correctly:

In [ ]:
quant_tables = torch.load(SAVE_PATH)

# Accesso diretto, già tensori
lum = quant_tables['step10_luminance']#.to('cuda')
chrom = quant_tables['step10_chrominance']#.to('cuda')

In [ ]:
print(f"Luminance:\n{lum}")
print(f"Chrominance:\n{chrom}")